# BCFMS Data Exploration

Notebook to update the IPA graph. Used to explore how to update a
graph.

Outputs are not committed to git (stripped by nbstripout).

## Django Setup

This cell must be run before importing any BCRHP models.

In [ ]:
import os
import django
from dotenv import load_dotenv
import sys

# Add the project root to sys.path so 'bcfms' is importable as a package
sys.path.insert(0, "/web_root/bcfms")

# Load environment variables from nr-bcap/.env
load_dotenv(dotenv_path=".env")

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "bcfms.settings")
django.setup()

### Update Photographs node to be 1:n

This
1. Gets or creates a draft of the IPA resource model
2. updates the nodegroup cardinality of the submission_photographs node.
3. Promotes the draft to the active graph.
4. Publishes the updated graph.


In [ ]:
from arches.app.models import models
import os
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
# First update the graph to make submission photos 1:n
source_graph = models.Graph.objects.get(slug="project_assessment", source_identifier__isnull=True)
draft_graph = source_graph.draft.first()
print(f"Got draft {draft_graph}")
if not draft_graph:
    print(f"no draft, creating one")
    draft_graph = source_graph.create_draft_graph()
node_to_update = models.Node.objects.get(
    graph=draft_graph,
    alias="submission_photographs",
    is_immutable=False,
)

node_to_update.nodegroup.cardinality = "n"
node_to_update.nodegroup.save()
source_graph.promote_draft_graph_to_active_graph()
source_graph.publish(notes="140 - Fixed nodegroup cardinality to make 1:n.")


### Update resource instances to use the current IPA graph

This moves all resource instances to the currently activated graph. This can be used on
the reverse migration too. It just finds the currently activate graph and moves all the
resources to it. If there are any tile data changes that need to be made as part of the
change they would need to be handled here too.

In [ ]:
from arches.app.models import models
import os
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

target_graph = models.Graph.objects.get(slug="project_assessment", source_identifier__isnull=True)
instances = models.ResourceInstance.objects.filter(graph=target_graph).exclude(graph_publication_id=target_graph.publication_id)
print(f"Updating {instances.count()} instances")
(models.ResourceInstance.objects.filter(graph=target_graph)
    .exclude(graph_publication_id=target_graph.publication_id)
    .update(graph_publication_id=target_graph.publication_id))
print("Instances updated.")


### Set active graph to graph before the newly created graph
Checks to see if the created graph is the latest version. If not, fails. If it is, sets
the previous graph to be the latest version.


In [ ]:
from arches.app.models import models
import os
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

current_graph = models.Graph.objects.get(slug="project_assessment", source_identifier__isnull=True)
if current_graph.publication.notes == "140 - Fixed nodegroup cardinality to make 1:n.":
    print(f"Setting {current_graph} to be the latest version")
    published_graphs = models.GraphXPublishedGraph.objects.filter(graph=current_graph).order_by("-published_time")
    print(f"Found {published_graphs.count()} published graphs")
    if published_graphs[0].notes == "140 - Fixed nodegroup cardinality to make 1:n.":
        print(f"Generated graph is the latest version")
        print(f"{published_graphs[1].graph.is_active}")
        print(f"{published_graphs[1].notes}")
        published = models.PublishedGraph.objects.get(publication_id=published_graphs[1].publicationid)
        serialized_graph = published.serialized_graph
        graph = models.Graph.objects.get(graphid=published_graphs[1].graph.graphid)
        graph.restore_state_from_serialized_graph(serialized_graph=serialized_graph)


### Update resource instances to use the previous IPA graph

This moves all resource instances to the currently activated graph. This can be used on
the reverse migration too. It just finds the currently activate graph and moves all the
resources to it. If there are any tile data changes that need to be made as part of the
change they would need to be handled here too.

In [ ]:
from arches.app.models import models
import os
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

target_graph = models.Graph.objects.get(slug="project_assessment", source_identifier__isnull=True)
instances = models.ResourceInstance.objects.filter(graph=target_graph).exclude(graph_publication_id=target_graph.publication_id)
print(f"Updating {instances.count()} instances")
models.ResourceInstance.objects.filter(graph=target_graph).exclude(graph_publication_id=target_graph.publication_id).update(graph_publication_id=target_graph.publication_id)
print("Instances updated.")


### Delete Graph created for the 1:m update

Uses the notes field to identify the published graph to be deleted. This fails if there are
any resources still attached to that graph

In [ ]:
from arches.app.models import models
import os
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

published_graph = models.GraphXPublishedGraph.objects.get(notes="140 - Fixed nodegroup cardinality to make 1:n.")
if published_graph:
    print(f"Deleting {published_graph}")
    published_graph.delete()
